# Predicción del volumen de lactosuero por lote

Notebook para Google Colab: carga y limpieza, análisis exploratorio, comparación cronológica de modelos de regresión y una función interactiva para estimar el volumen de lactosuero (L).

**Dependencias:** pandas, numpy, scikit-learn y matplotlib. El conjunto tiene pocos registros (81 filas), por lo que las métricas deben interpretarse como una evaluación inicial, no como garantía de rendimiento futuro.

## Celda 1. Importar librerías y configurar parámetros

Se define el mínimo de lotes, el umbral aceptable de MAPE y la semilla reproducible.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error, r2_score, mean_absolute_error

MIN_LOTES = 30
UMBRAL_MAPE = 20.0  # Porcentaje; ajustar según las necesidades del proceso
SEMILLA = 42

pd.set_option("display.max_columns", None)


## Celda 2. Cargar el archivo CSV

En Colab, sube `dataset_sintetico_lactosuero.csv` al panel **Archivos**. Si el archivo tiene otro nombre o ubicación, modifica `RUTA_CSV`.

In [ ]:
RUTA_CSV = "/content/dataset_sintetico_lactosuero.csv"

# Se intenta leer con UTF-8; si el CSV fue guardado con otra codificación,
# puede probarse encoding="latin-1".
df_original = pd.read_csv(RUTA_CSV, encoding="utf-8")
print("Dimensiones originales:", df_original.shape)
display(df_original.head())
print("\\nTipos de datos:")
print(df_original.dtypes)


## Celda 3. Diagnosticar y limpiar los datos

Convierte fechas y variables numéricas, elimina duplicados y registros sin fecha o sin variable objetivo. Los valores numéricos faltantes de las variables predictoras se imputarán dentro de cada modelo para evitar fuga de información.

In [ ]:
df = df_original.copy()
df.columns = df.columns.str.strip()

# Quitar duplicados exactos.
df = df.drop_duplicates()

# Normalizar fecha.
if "fecha" not in df.columns:
    raise ValueError("El archivo debe contener la columna 'fecha'.")
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce", dayfirst=True)

# Columnas que se esperan en este conjunto de datos.
objetivo = "volumen_lactosuero_L"
columnas_requeridas = [
    "fecha", "tipo_producto", "volumen_leche_L",
    "grasa_leche_pct", "proteina_leche_pct", "solidos_totales_pct",
    "temperatura_proceso_C", "rendimiento_historico_movil_pct",
    "agua_limpieza_L", objetivo
]
faltantes = [c for c in columnas_requeridas if c not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas requeridas en el CSV: {faltantes}")

# Convertir a numérico cualquier campo cuantitativo esperado.
columnas_numericas = [
    "mes", "volumen_leche_L", "grasa_leche_pct", "proteina_leche_pct",
    "solidos_totales_pct", "temperatura_proceso_C",
    "rendimiento_historico_movil_pct", "rendimiento_real_lote_pct",
    "agua_limpieza_L", objetivo
]
for col in columnas_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# No se puede entrenar sin fecha ni variable objetivo.
df = df.dropna(subset=["fecha", objetivo])
# Eliminar objetivos físicamente imposibles (volumen negativo).
df = df[df[objetivo] >= 0].copy()

# Crear variables de calendario a partir de la fecha.
df["anio"] = df["fecha"].dt.year
df["dia_del_anio"] = df["fecha"].dt.dayofyear
df["dia_semana"] = df["fecha"].dt.dayofweek

# Orden temporal para respetar el flujo del algoritmo.
df = df.sort_values("fecha").reset_index(drop=True)

print("Dimensiones tras limpieza:", df.shape)
print("Filas eliminadas:", len(df_original) - len(df))
print("\\nValores faltantes por columna:")
display(df.isna().sum().to_frame("faltantes"))
display(df.head())

if len(df) < MIN_LOTES:
    raise ValueError(
        f"Hay {len(df)} lotes válidos y se requieren al menos {MIN_LOTES}. "
        "Recolecta más datos o reduce justificadamente MIN_LOTES."
    )


## Celda 4. Análisis exploratorio de datos (EDA)

Se revisan estadísticas descriptivas, distribución del volumen de lactosuero y evolución temporal. Las gráficas ayudan a detectar tendencias y valores atípicos; no prueban causalidad.

In [ ]:
display(df.describe(include="all").T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df[objetivo].dropna(), bins=15, edgecolor="black")
axes[0].set_title("Distribución del volumen de lactosuero")
axes[0].set_xlabel("Volumen (L)")
axes[0].set_ylabel("Frecuencia")

axes[1].plot(df["fecha"], df[objetivo], marker="o", linewidth=1)
axes[1].set_title("Volumen por fecha")
axes[1].set_xlabel("Fecha")
axes[1].set_ylabel("Volumen (L)")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

# Correlaciones solo entre columnas numéricas.
corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(10, 7))
plt.imshow(corr, aspect="auto", vmin=-1, vmax=1)
plt.colorbar(label="Correlación de Pearson")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Matriz de correlación (variables numéricas)")
plt.tight_layout()
plt.show()


## Celda 5. Partición cronológica y entrenamiento

Se reserva el tramo más reciente para prueba y el tramo anterior para validación. Se comparan regresión lineal y bosque aleatorio. La selección se hace únicamente con el MAPE de validación, no con el conjunto de prueba.

In [ ]:
# Variables de entrada. Se excluyen identificadores y el rendimiento real del lote
# para evitar usar información que podría no estar disponible al hacer la predicción.
columnas_features = [
    "mes", "tipo_producto", "volumen_leche_L", "grasa_leche_pct",
    "proteina_leche_pct", "solidos_totales_pct", "temperatura_proceso_C",
    "rendimiento_historico_movil_pct", "agua_limpieza_L",
    "anio", "dia_del_anio", "dia_semana"
]
columnas_features = [c for c in columnas_features if c in df.columns]

X = df[columnas_features].copy()
y = df[objetivo].copy()

n = len(df)
i_train = int(n * 0.60)
i_valid = int(n * 0.80)
if i_train < 2 or i_valid <= i_train or i_valid >= n:
    raise ValueError("No hay suficientes datos para una partición 60/20/20.")

X_train, y_train = X.iloc[:i_train], y.iloc[:i_train]
X_valid, y_valid = X.iloc[i_train:i_valid], y.iloc[i_train:i_valid]
X_test, y_test = X.iloc[i_valid:], y.iloc[i_valid:]

columnas_cat = X.select_dtypes(include=["object", "category"]).columns.tolist()
columnas_num = [c for c in X.columns if c not in columnas_cat]

preprocesador = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputar", SimpleImputer(strategy="median")),
            ("escalar", StandardScaler())
        ]), columnas_num),
        ("cat", Pipeline([
            ("imputar", SimpleImputer(strategy="most_frequent")),
            ("codificar", OneHotEncoder(handle_unknown="ignore"))
        ]), columnas_cat)
    ],
    remainder="drop"
)

modelos = {
    "Regresión lineal": LinearRegression(),
    "Bosque aleatorio": RandomForestRegressor(
        n_estimators=300, min_samples_leaf=2, random_state=SEMILLA
    )
}

resultados_validacion = []
pipelines = {}

for nombre, estimador in modelos.items():
    pipe = Pipeline([
        ("preprocesamiento", preprocesador),
        ("modelo", estimador)
    ])
    pipe.fit(X_train, y_train)
    pred_valid = np.maximum(pipe.predict(X_valid), 0)
    # MAPE: proteger contra valores reales iguales a cero.
    mape = mean_absolute_percentage_error(
        np.maximum(y_valid.to_numpy(), 1e-8),
        np.maximum(pred_valid, 1e-8)
    ) * 100
    resultados_validacion.append({"modelo": nombre, "MAPE_validacion_pct": mape})
    pipelines[nombre] = pipe

tabla_validacion = pd.DataFrame(resultados_validacion).sort_values("MAPE_validacion_pct")
display(tabla_validacion)

mejor_nombre = tabla_validacion.iloc[0]["modelo"]
mejor_pipeline = pipelines[mejor_nombre]
print("Modelo seleccionado:", mejor_nombre)


## Celda 6. Evaluar en prueba y comprobar el umbral

El conjunto de prueba simula datos futuros y se utiliza una sola vez para evaluar el modelo elegido. Si el MAPE supera el umbral, revisa calidad de datos, variables, frecuencia de registros y modelos; no ajustes repetidamente mirando el conjunto de prueba.

In [ ]:
pred_test = np.maximum(mejor_pipeline.predict(X_test), 0)
mape_test = mean_absolute_percentage_error(
    np.maximum(y_test.to_numpy(), 1e-8),
    np.maximum(pred_test, 1e-8)
) * 100
r2_test = r2_score(y_test, pred_test) if len(y_test) >= 2 else np.nan
mae_test = mean_absolute_error(y_test, pred_test)

print(f"Modelo: {mejor_nombre}")
print(f"MAPE de prueba: {mape_test:.2f}%")
print(f"MAE de prueba: {mae_test:.2f} L")
print(f"R² de prueba: {r2_test:.3f}")
print(f"Umbral MAPE: {UMBRAL_MAPE:.2f}%")

if mape_test <= UMBRAL_MAPE:
    print("Resultado: el MAPE de prueba está dentro del umbral configurado.")
else:
    print("Resultado: MAPE sobre el umbral. Revisar variables, datos y modelos.")

comparacion = pd.DataFrame({
    "fecha": df.iloc[i_valid:]["fecha"].to_numpy(),
    "real_L": y_test.to_numpy(),
    "predicho_L": pred_test
})
display(comparacion)

plt.figure(figsize=(10, 4))
plt.plot(comparacion["fecha"], comparacion["real_L"], marker="o", label="Real")
plt.plot(comparacion["fecha"], comparacion["predicho_L"], marker="x", label="Predicción")
plt.title("Prueba: volumen real frente a predicho")
plt.xlabel("Fecha")
plt.ylabel("Volumen de lactosuero (L)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()


## Celda 7. Función para recibir datos del usuario

La función recibe un diccionario con las variables de entrada y devuelve el volumen estimado en litros. Los campos no incluidos se imputan con las reglas aprendidas durante el entrenamiento. Para obtener una estimación útil, introduce valores en las mismas unidades y rangos que los registros históricos.

In [ ]:
def predecir_lactosuero(
    mes,
    tipo_producto,
    volumen_leche_L,
    grasa_leche_pct,
    proteina_leche_pct,
    solidos_totales_pct,
    temperatura_proceso_C,
    rendimiento_historico_movil_pct,
    agua_limpieza_L,
    fecha=None
):
    """Estima litros de lactosuero para un lote a partir de variables de proceso."""
    fecha_pred = pd.to_datetime(fecha) if fecha else df["fecha"].max() + pd.Timedelta(days=1)

    entrada = {
        "mes": mes,
        "tipo_producto": tipo_producto,
        "volumen_leche_L": volumen_leche_L,
        "grasa_leche_pct": grasa_leche_pct,
        "proteina_leche_pct": proteina_leche_pct,
        "solidos_totales_pct": solidos_totales_pct,
        "temperatura_proceso_C": temperatura_proceso_C,
        "rendimiento_historico_movil_pct": rendimiento_historico_movil_pct,
        "agua_limpieza_L": agua_limpieza_L,
        "anio": fecha_pred.year,
        "dia_del_anio": fecha_pred.dayofyear,
        "dia_semana": fecha_pred.dayofweek
    }
    fila = pd.DataFrame([{c: entrada.get(c, np.nan) for c in columnas_features}])
    estimacion = float(mejor_pipeline.predict(fila)[0])
    return max(0.0, estimacion)

# Ejemplo editable: sustituye los valores por los del próximo lote.
ejemplo = predecir_lactosuero(
    mes=10,
    tipo_producto="queso fresco",
    volumen_leche_L=500,
    grasa_leche_pct=3.5,
    proteina_leche_pct=3.2,
    solidos_totales_pct=12.5,
    temperatura_proceso_C=35,
    rendimiento_historico_movil_pct=10,
    agua_limpieza_L=100,
    fecha="2026-10-23"
)
print(f"Volumen estimado de lactosuero: {ejemplo:.2f} L")


## Celda 8. Entrada interactiva por teclado (opcional)

Al ejecutar esta celda, Colab solicitará los datos del lote. Usa exactamente una categoría de producto coherente con el histórico. Si hay categorías distintas, consulta `df['tipo_producto'].unique()`.

In [ ]:
print("Tipos de producto encontrados:", df["tipo_producto"].dropna().unique())

try:
    resultado_usuario = predecir_lactosuero(
        mes=int(input("Mes (1-12): ")),
        tipo_producto=input("Tipo de producto: ").strip(),
        volumen_leche_L=float(input("Volumen de leche (L): ")),
        grasa_leche_pct=float(input("Grasa de leche (%): ")),
        proteina_leche_pct=float(input("Proteína de leche (%): ")),
        solidos_totales_pct=float(input("Sólidos totales (%): ")),
        temperatura_proceso_C=float(input("Temperatura de proceso (°C): ")),
        rendimiento_historico_movil_pct=float(input("Rendimiento histórico móvil (%): ")),
        agua_limpieza_L=float(input("Agua de limpieza (L): ")),
        fecha=input("Fecha del lote (AAAA-MM-DD; Enter para fecha siguiente): ").strip() or None
    )
    print(f"\\nEstimación: {resultado_usuario:.2f} litros de lactosuero")
except ValueError as error:
    print("Entrada no válida. Verifica que las fechas y valores numéricos sean correctos.", error)


## Celda 9. Límites y condiciones de uso

- El archivo de ejemplo contiene pocos lotes; una división temporal puede producir métricas inestables.
- El conjunto sintético no demuestra que el modelo funcione con una quesería real. Valida con datos reales, consistentes y representativos.
- MAPE puede ser engañoso cuando los valores reales son cero o muy pequeños; por eso se protege el cálculo con un valor mínimo. Complementa con MAE y R².
- La partición cronológica reduce el riesgo de fuga temporal, pero las variables deben estar disponibles antes de conocer el resultado del lote.
- La predicción estima volumen de lactosuero, no el valor económico ni la recuperación de lactosa/proteína. Para un balance de masa se necesitarían mediciones de composición y caudales adicionales.
- Registra unidades, calibra instrumentos y documenta cambios de proceso. No uses la predicción como sustituto de mediciones críticas.